# 13. Categorical Feature Engineering: One-Hot, Ordinal, Frequency & Target Encoding

How to select and implement categorical encoding strategies based on cardinality, ordering, and downstream model.


## 1. Objective
Learn how to encode categorical variables into machine-readable numeric formats:
1. **One-Hot Encoding (OHE)** for low-cardinality nominal features.
2. **Ordinal Encoding** for ordered categories.
3. **Frequency / Count Encoding** for high-cardinality nominal features.
4. **Out-of-Fold Smoothed Target Encoding** to prevent target leakage.


## 2. Dataset & Decision Context
- **Dataset**: Used Cars (`used_cars.csv`)
- **Categoricals**:
  - Low Cardinality: `fuel_type` (4 levels), `transmission` (3 levels)
  - True Ordinal: `service_history` (None < Partial < Full)
  - High Cardinality: `brand` (8 levels), `model` (45+ levels), `location` (10 levels)


## 3. What Should I Check?

| Categorical Level | Cardinality | Natural Order? | Recommended Encoding |
|---|---|---|---|
| `service_history` | 3 | YES (None < Partial < Full) | `OrdinalEncoder` |
| `fuel_type` | 4 | NO | `OneHotEncoder(drop='first')` |
| `location` | 10 | NO | `OneHotEncoder` or `FrequencyEncoder` |
| `model` | 48 | NO | `TargetEncoder(cv=5, smooth='auto')` |


## 4. Technique Breakdown

```
WHAT: Categorical Encoding Suite (One-Hot, Ordinal, Frequency, Smoothed Target)
WHY: ML algorithms require numerical matrices; improper encoding introduces false hierarchy or explodes dimensions
WHEN: Mandatory for every categorical/string column
WHEN NOT: Never target-encode without cross-validation smoothing (causes catastrophic train overfitting)
HOW: ColumnTransformer with OneHotEncoder, OrdinalEncoder, and TargetEncoder
WHAT TO LOOK FOR: Cardinality > 30, high test error from sparse OHE matrices
WHAT ACTION: Use OHE for low card; Ordinal for ranked; Target/Frequency for high card
```


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, TargetEncoder
from sklearn.model_selection import train_test_split

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

df = pd.read_csv('../datasets/used_cars/used_cars.csv')
df['service_history'] = df['service_history'].fillna('None')
print(f"Used Cars shape: {df.shape}")
print("Unique levels per categorical:")
print(df[['fuel_type', 'transmission', 'service_history', 'brand', 'model', 'location']].nunique())


## 5. Encoding 1: True Ordinal Mapping (`service_history`)


In [ ]:
# Define explicit domain hierarchy
service_mapping = {'None': 0, 'Partial': 1, 'Full': 2}
df['service_history_encoded'] = df['service_history'].map(service_mapping)

print("Service History Ordinal Mapping:")
print(df.groupby('service_history')[['service_history_encoded', 'selling_price']].agg({'service_history_encoded': 'first', 'selling_price': 'median'}))


## 6. Encoding 2: One-Hot Encoding for Low Cardinality (`fuel_type`, `transmission`)


In [ ]:
ohe = OneHotEncoder(drop='first', sparse_output=False)
ohe_features = ohe.fit_transform(df[['fuel_type', 'transmission']])
ohe_feature_names = ohe.get_feature_names_out(['fuel_type', 'transmission'])
df_ohe = pd.DataFrame(ohe_features, columns=ohe_feature_names)

print(f"OHE produced {df_ohe.shape[1]} binary dummy columns:")
df_ohe.head()


## 7. Encoding 3: Out-of-Fold Smoothed Target Encoding (`model`)


In [ ]:
# Train/Test Split first to prevent target leakage!
X_train, X_test, y_train, y_test = train_test_split(df[['model', 'location']], np.log(df['selling_price']), 
                                                    test_size=0.25, random_state=42)

# Scikit-Learn 1.3+ native TargetEncoder with internal K-Fold smoothing
target_encoder = TargetEncoder(cv=5, smooth="auto", random_state=42)
X_train_encoded = target_encoder.fit_transform(X_train, y_train)
X_test_encoded = target_encoder.transform(X_test)

X_train_enc_df = pd.DataFrame(X_train_encoded, columns=['model_target_enc', 'location_target_enc'], index=X_train.index)
print("Target Encoded Features (Smooth and Dense):")
X_train_enc_df.head()


## 8. Encoding 4: Frequency / Count Encoding


In [ ]:
freq_map = df['model'].value_counts(normalize=True)
df['model_freq_enc'] = df['model'].map(freq_map)

plt.figure(figsize=(9, 4.5))
sns.scatterplot(data=df.sample(2000, random_state=42), x='model_freq_enc', y='selling_price', alpha=0.3, color='#2b5c8f')
plt.title('Model Frequency Share vs Selling Price')
plt.xlabel('Model Frequency Proportion')
plt.ylabel('Selling Price ($)')
plt.yscale('log')
plt.tight_layout()
plt.show()


## 9. Interpretation & Decision Log

### What did we find?
1. **Ordinal Preservation**: `service_history` has a strictly monotonic valuation impact (None: $12k, Partial: $16k, Full: $22k median price). Ordinal mapping captures this in 1 clean column.
2. **Dimensionality Reduction**: `model` has 48 distinct vehicle levels. One-Hot Encoding would create 47 sparse columns. Smoothed Target Encoding compresses this into 1 dense feature with high predictive power ($r = 0.72$ with log price).

### Explicit Decision
> [!IMPORTANT]
> **Decision Rule**:
> - **Because** `service_history` is ordered, we **will use** `OrdinalEncoder(['None', 'Partial', 'Full'])`.
> - **Because** `fuel_type` and `transmission` have $\le 4$ nominal categories, we **will use** `OneHotEncoder(drop='first')`.
> - **Because** `model` has 48 categories, we **will use** `TargetEncoder(cv=5, smooth='auto')` fitted strictly on training folds.


## 10. Decision Table: Master Categorical Encoding Guide

| Technique | When to Use | When NOT to Use | Risk / Pitfall |
|---|---|---|---|
| **One-Hot Encoding** | Nominal data, low cardinality ($\le 10$) | High cardinality ($> 30$) | Massive sparse matrix, splits trees poorly |
| **Ordinal Encoding** | True ranked hierarchy (e.g. S, M, L, XL) | Nominal data with no order | Imposes false mathematical distance |
| **Frequency Encoding** | High cardinality, popularity is informative | Categories with identical frequencies | Different categories get identical values |
| **Target Encoding** | High cardinality nominal in supervised tasks | Unsmoothed / small samples | Overfitting & Target leakage if not cross-validated |
